In [1]:
import os
import glob
import ee
import pandas as pd
import rasterio
from IPython.display import Image, display

PROJECT = 'gee-mb-06'

WORKDIR = os.path.join(os.path.expanduser('~'), 'co2_project')
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(os.getcwd())

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

C:\Users\mayar\co2_project


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [2]:
aoi = ee.Geometry.Rectangle([8.62, 36.72, 8.78, 36.84])
CRS = 'EPSG:32632'
T1 = ('2020-06-01', '2020-09-30')
T2 = ('2025-06-01', '2025-09-30')
YEARS = 5

In [3]:
def mask_scl(img):
    scl = img.select('SCL')
    ok = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
    return img.updateMask(ok).select(['B2', 'B3', 'B4', 'B8']).divide(10000)

def composite(period):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(*period)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
            .map(mask_scl)
            .median()
            .clip(aoi))

def add_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = img.normalizedDifference(['B3', 'B8']).rename('NDWI')
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': img.select('B8'), 'RED': img.select('B4'), 'BLUE': img.select('B2')}
    ).rename('EVI')
    return img.addBands([ndvi, ndwi, evi]).updateMask(ndwi.lt(0))

s1 = add_indices(composite(T1))
s2 = add_indices(composite(T2))

landcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(aoi)
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi).rename('elevation')
slope = ee.Terrain.slope(dem).rename('slope')

stack1 = s1.addBands([landcover.rename('landcover'), dem, slope]).toFloat()
stack2 = s2.addBands([landcover.rename('landcover'), dem, slope]).toFloat()

In [4]:
for name, img in [('t1', stack1), ('t2', stack2)]:
    ee.batch.Export.image.toDrive(
        image=img,
        description=f'stack_{name}',
        folder='gee_exports',
        region=aoi,
        scale=10,
        crs=CRS,
        maxPixels=1e13
    ).start()

In [5]:
for t in ee.batch.Task.list()[:2]:
    print(t.config['description'], t.status()['state'])

stack_t2 READY
stack_t1 READY


In [6]:
elev = dem.reduceRegion(
    reducer=ee.Reducer.min().combine(ee.Reducer.max(), sharedInputs=True).combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=aoi, scale=30, crs=CRS, maxPixels=1e13
).getInfo()
print(elev)

share_800 = dem.gte(800).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=30, crs=CRS, maxPixels=1e13
).getInfo()['elevation']
print('share of AOI at or above 800 m:', round(100 * share_800, 1), '%')

{'elevation_max': 1014, 'elevation_mean': 536.8907149961008, 'elevation_min': 90}
share of AOI at or above 800 m: 5.8 %


In [7]:
def show(img, band, vmin, vmax, palette):
    url = img.select(band).getThumbURL({
        'region': aoi,
        'dimensions': 600,
        'min': vmin,
        'max': vmax,
        'palette': palette
    })
    display(Image(url=url))

green = ['#a50026', '#f46d43', '#fee08b', '#a6d96a', '#1a9850', '#004d26']
show(s1, 'NDVI', 0, 1, green)
show(s2, 'NDVI', 0, 1, green)
show(s1, 'EVI', 0, 1, green)
show(s2, 'EVI', 0, 1, green)
show(s2, 'NDWI', -1, 1, ['#8c510a', '#f6e8c3', '#c7eae5', '#01665e'])
show(landcover, 'Map', 10, 100, ['#006400', '#ffbb22', '#ffff4c', '#f096ff', '#fa0000', '#b4b4b4', '#f0f0f0', '#0064c8', '#0096a0', '#00cf75', '#fae6a0'])
show(dem, 'elevation', 0, 1100, ['#1a9850', '#fee08b', '#d73027', '#ffffff'])

In [8]:
names = {10: 'tree', 20: 'shrub', 30: 'grass', 40: 'crop', 50: 'built', 60: 'bare', 70: 'snow', 80: 'water', 90: 'wetland', 95: 'mangrove', 100: 'moss'}

hist = landcover.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(), geometry=aoi, scale=10, crs=CRS, maxPixels=1e13
).getInfo()['Map']
total_px = sum(hist.values())
shares = pd.Series({names[int(float(k))]: round(100 * v / total_px, 1) for k, v in hist.items()})
print(shares.sort_values(ascending=False).to_string())

for code in [10, 20, 30, 40, 60]:
    if names[code] in shares.index:
        val = (s2.select('NDVI')
               .updateMask(landcover.eq(code))
               .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=10, crs=CRS, maxPixels=1e13)
               .getInfo())
        print(names[code], val)

tree     72.4
grass    20.0
shrub     4.2
built     1.7
water     0.9
crop      0.8
bare      0.1
tree {'NDVI': 0.7499452357732772}
shrub {'NDVI': 0.5155889289178844}
grass {'NDVI': 0.38648803787644426}
crop {'NDVI': 0.23342276529156464}
bare {'NDVI': 0.1395283831994468}


In [9]:
TREE = (65.2, 35.1)
ELEV_SPLIT = 800

AGB = {10: TREE[0], 20: 31.5, 30: 6.2, 40: 10.0, 50: 0.0, 60: 0.0, 70: 0.0, 80: 0.0, 90: 6.2, 95: 0.0, 100: 0.0}

SOURCES = {
    'tree (10), below 800 m': '65.2 t d.m./ha, subtropical dry forest, Africa, IPCC 2019 Refinement Table 4.7 (Rozendaal et al. 2022, Environ. Res. Lett. 17:014047)',
    'tree (10), 800 m and above': '35.1 t d.m./ha, subtropical montane forest, Africa, same source, FAO GEZ mountain zone starts at about 800 m',
    'cropland (40)': '10 t d.m./ha, IPCC default for annual cropland',
    'grass (30), wetland (90)': '6.2 t d.m./ha, IPCC 2006 Table 6.4 value, climate zone not verified for this area',
    'shrub (20)': '31.5 t d.m./ha, midpoint of 9 to 54 t/ha reported for maquis understorey in Kroumirie cork oak stands (Ann. For. Sci. 2001), units not verified',
    'built, bare, snow, water, mangrove, moss': 'set to 0'
}

CARBON_FRACTION = 0.47
CO2_FACTOR = 44 / 12
ROOT_RATIO = 0.0
AMP = 0.2

def agb(s, mult=1.0, amp=AMP, tree=TREE):
    keys = list(AGB.keys())
    vals = [AGB[k] * mult for k in keys]
    tree_img = ee.Image(tree[0]).where(dem.gte(ELEV_SPLIT), tree[1]).multiply(mult)
    base = landcover.remap(keys, vals).where(landcover.eq(10), tree_img)
    u = s.select('NDVI').unitScale(0.2, 0.9).clamp(0, 1)
    return base.multiply(u.multiply(2 * amp).add(1 - amp)).rename('AGB')

def co2e(s, mult=1.0, amp=AMP, tree=TREE):
    factor = (1 + ROOT_RATIO) * CARBON_FRACTION * CO2_FACTOR
    return agb(s, mult, amp, tree).multiply(factor).rename('CO2e')

def scenario(mult=1.0, amp=AMP, tree=TREE):
    c1 = co2e(s1, mult, amp, tree)
    c2 = co2e(s2, mult, amp, tree)
    valid = c1.mask().And(c2.mask())
    c1 = c1.updateMask(valid)
    c2 = c2.updateMask(valid)
    return ee.Image.cat([c1.rename('t1'), c2.rename('t2'), c2.subtract(c1).rename('net')])

def totals(img):
    area_ha = ee.Image.pixelArea().divide(10000)
    img = img.addBands(img.select('t1').multiply(0).add(1).rename('area'))
    return img.multiply(area_ha).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=10, crs=CRS, maxPixels=1e13
    ).getInfo()

In [10]:
pipe_mean = agb(s1).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=10, crs=CRS, maxPixels=1e13
).getInfo()['AGB']

gedi_mean = ee.Image('LARSE/GEDI/GEDI04_B_002').select('MU').reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e13
).getInfo()['MU']

CAL = gedi_mean / pipe_mean

print('Pipeline mean AGB, 2020 composite (t/ha):', round(pipe_mean, 1))
print('GEDI L4B mean AGB, Apr 2019 to Aug 2021 (t/ha):', round(gedi_mean, 1))
print('Calibration multiplier:', round(CAL, 3))

Pipeline mean AGB, 2020 composite (t/ha): 53.5
GEDI L4B mean AGB, Apr 2019 to Aug 2021 (t/ha): 62.3
Calibration multiplier: 1.164


In [11]:
r_ipcc = scenario(1.0)
r_gedi = scenario(CAL)

show(agb(s1), 'AGB', 0, 80, green)
show(agb(s2), 'AGB', 0, 80, green)
show(r_ipcc, 't1', 0, 120, green)
show(r_ipcc, 't2', 0, 120, green)
show(r_ipcc, 'net', -10, 10, ['#b2182b', '#f7f7f7', '#2166ac'])

In [12]:
res_i = totals(r_ipcc)
res_g = totals(r_gedi)

def col(r):
    return [
        r['area'],
        r['t1'],
        r['t2'],
        r['net'],
        r['net'] / YEARS,
        r['net'] / r['area'],
        r['net'] / r['area'] / YEARS
    ]

summary = pd.DataFrame({
    'metric': ['valid area', 'CO2e stock t1', 'CO2e stock t2', 'net CO2e change', 'annual CO2e rate', 'change per hectare', 'change per hectare per year'],
    'unit': ['ha', 't CO2e', 't CO2e', 't CO2e', 't CO2e/yr', 't CO2e/ha', 't CO2e/ha/yr'],
    'ipcc_values': col(res_i),
    'gedi_calibrated': col(res_g)
}).round(1)

summary.to_csv('summary.csv', index=False)
print(summary.to_string(index=False))

                     metric         unit  ipcc_values  gedi_calibrated
                 valid area           ha      18771.1          18771.1
              CO2e stock t1       t CO2e    1733800.4        2018006.2
              CO2e stock t2       t CO2e    1745732.9        2031894.8
            net CO2e change       t CO2e      11932.5          13888.5
           annual CO2e rate    t CO2e/yr       2386.5           2777.7
         change per hectare    t CO2e/ha          0.6              0.7
change per hectare per year t CO2e/ha/yr          0.1              0.1


In [13]:
rows = []
for label, base in [('ipcc_values', 1.0), ('gedi_calibrated', CAL)]:
    for f in [0.8, 1.0, 1.2]:
        r = totals(scenario(base * f))
        rows.append({
            'scenario': label,
            'multiplier': round(base * f, 3),
            'stock_t1_tCO2e': round(r['t1']),
            'stock_t2_tCO2e': round(r['t2']),
            'net_change_tCO2e': round(r['net'])
        })

sens_mult = pd.DataFrame(rows)
sens_mult.to_csv('sensitivity_multiplier.csv', index=False)
print(sens_mult.to_string(index=False))

rows = []
for a in [0.1, 0.2, 0.3]:
    r = totals(scenario(CAL, a))
    rows.append({
        'ndvi_modifier_amplitude': a,
        'net_change_tCO2e': round(r['net'])
    })

sens_amp = pd.DataFrame(rows)
sens_amp.to_csv('sensitivity_ndvi_modifier.csv', index=False)
print(sens_amp.to_string(index=False))

rows = []
for label, tr in [('all_dry_forest_65.2', (65.2, 65.2)), ('elevation_split_65.2_35.1', (65.2, 35.1)), ('all_montane_35.1', (35.1, 35.1))]:
    mean_agb = agb(s1, 1.0, AMP, tr).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, crs=CRS, maxPixels=1e13
    ).getInfo()['AGB']
    r = totals(scenario(1.0, AMP, tr))
    rows.append({
        'tree_zone_choice': label,
        'mean_AGB_2020_t_ha': round(mean_agb, 1),
        'stock_t1_tCO2e': round(r['t1']),
        'net_change_tCO2e': round(r['net'])
    })

sens_zone = pd.DataFrame(rows)
sens_zone.to_csv('sensitivity_forest_zone.csv', index=False)
print(sens_zone.to_string(index=False))

       scenario  multiplier  stock_t1_tCO2e  stock_t2_tCO2e  net_change_tCO2e
    ipcc_values       0.800         1387040         1396586              9546
    ipcc_values       1.000         1733800         1745733             11933
    ipcc_values       1.200         2080560         2094880             14319
gedi_calibrated       0.931         1614405         1625516             11111
gedi_calibrated       1.164         2018006         2031895             13889
gedi_calibrated       1.397         2421607         2438274             16666
 ndvi_modifier_amplitude  net_change_tCO2e
                     0.1              6944
                     0.2             13889
                     0.3             20833
         tree_zone_choice  mean_AGB_2020_t_ha  stock_t1_tCO2e  net_change_tCO2e
      all_dry_forest_65.2                55.3         1791498             12222
elevation_split_65.2_35.1                53.5         1733800             11933
         all_montane_35.1                3

In [14]:
paths = {}
for name in ['stack_t1.tif', 'stack_t2.tif']:
    found = glob.glob(os.path.join(os.path.expanduser('~'), 'Downloads', name)) + glob.glob(os.path.join(WORKDIR, name))
    paths[name] = found[0] if found else None
print(paths)

for name, p in paths.items():
    if p:
        with rasterio.open(p) as src:
            print(name, src.shape, src.crs, src.count, src.res)
            print(src.descriptions)

{'stack_t1.tif': 'C:\\Users\\mayar\\Downloads\\stack_t1.tif', 'stack_t2.tif': 'C:\\Users\\mayar\\Downloads\\stack_t2.tif'}
stack_t1.tif (1337, 1433) EPSG:32632 10 (10.0, 10.0)
('B2', 'B3', 'B4', 'B8', 'NDVI', 'NDWI', 'EVI', 'landcover', 'elevation', 'slope')
stack_t2.tif (1337, 1433) EPSG:32632 10 (10.0, 10.0)
('B2', 'B3', 'B4', 'B8', 'NDVI', 'NDWI', 'EVI', 'landcover', 'elevation', 'slope')


In [15]:
source_lines = '\n'.join(f'- {k}: {v}' for k, v in SOURCES.items())

readme = f"""# Satellite-Based CO2 Absorption Estimation, Ain Draham (Kroumirie, NW Tunisia)

## Area
- Rectangle lon 8.62 to 8.78, lat 36.72 to 36.84, {res_i['area']:.0f} ha valid
- Evergreen cork oak and deciduous oak forest, with farmland and the town of Ain Draham

## Data sources
- Sentinel-2 L2A (COPERNICUS/S2_SR_HARMONIZED), SCL cloud mask, median seasonal composites
- ESA WorldCover v200 (10 m land cover)
- SRTM DEM (elevation, slope), 800 m split for the IPCC mountain zone
- GEDI L4B v2 (LARSE/GEDI/GEDI04_B_002, mean AGBD, 1 km, Apr 2019 to Aug 2021)

## Periods
- t1: {T1[0]} to {T1[1]}
- t2: {T2[0]} to {T2[1]}

## Method
- NDWI < 0 keeps land, water masked
- AGB = land cover lookup value x NDVI modifier (1 - {AMP} to 1 + {AMP})
- Carbon = AGB x {CARBON_FRACTION}
- CO2e = carbon x 44/12
- Sequestration = CO2e(t2) - CO2e(t1)

## Biomass lookup values (t d.m./ha)
{source_lines}

## GEDI comparison
- Pipeline mean AGB, 2020 composite: {pipe_mean:.1f} t/ha
- GEDI L4B mean AGB: {gedi_mean:.1f} t/ha
- Calibration multiplier: {CAL:.3f}
- GEDI period matches t1 only, not t2

## Results (t CO2e, {YEARS} years, {res_i['area']:.0f} ha)
- Lookup values as listed: net change {res_i['net']:.0f}
- GEDI-calibrated: net change {res_g['net']:.0f}

## Scope
- Above-ground biomass only, belowground biomass and soil carbon excluded
- No verified root ratios for these land classes

## Limitations
- No field plot validation
- IPCC defaults are continental averages by ecological zone, local cork oak stands can be much denser
- Land cover is fixed between periods, so change comes only from NDVI
- Net change depends directly on the NDVI modifier amplitude, see sensitivity_ndvi_modifier.csv
- The 800 m zone split is an approximation of the FAO ecological zone map
- Two summer composites only, drought, dieback, fire and seasonal differences can change NDVI without changing biomass
- Not compliant with registry MRV methodologies

## Outputs
- summary.csv
- sensitivity_multiplier.csv
- sensitivity_ndvi_modifier.csv
- sensitivity_forest_zone.csv
- stack_t1.tif, stack_t2.tif
"""

with open('README.md', 'w') as f:
    f.write(readme)

print(os.path.abspath('README.md'))

C:\Users\mayar\co2_project\README.md


In [16]:
import os
import pandas as pd

def val(name):
    return globals().get(name, 'missing')

s800 = round(100 * share_800, 1) if 'share_800' in globals() else 'missing'
lc = shares.sort_values(ascending=False).to_string() if 'shares' in globals() else 'missing'

parts = [
    'ELEVATION: ' + str(val('elev')),
    'SHARE_AT_OR_ABOVE_800M_PERCENT: ' + str(s800),
    'LAND_COVER_PERCENT:\n' + lc,
    'PIPELINE_MEAN_AGB_2020: ' + str(val('pipe_mean')),
    'GEDI_MEAN_AGB: ' + str(val('gedi_mean')),
    'CALIBRATION_MULTIPLIER: ' + str(val('CAL'))
]

for f in ['summary.csv', 'sensitivity_multiplier.csv', 'sensitivity_ndvi_modifier.csv', 'sensitivity_forest_zone.csv']:
    if os.path.exists(f):
        parts.append(f + ':\n' + pd.read_csv(f).to_string(index=False))
    else:
        parts.append(f + ': missing')

text = '\n\n'.join(parts)

with open('report_numbers.txt', 'w') as f:
    f.write(text)

print(text)

ELEVATION: {'elevation_max': 1014, 'elevation_mean': 536.8907149961008, 'elevation_min': 90}

SHARE_AT_OR_ABOVE_800M_PERCENT: 5.8

LAND_COVER_PERCENT:
tree     72.4
grass    20.0
shrub     4.2
built     1.7
water     0.9
crop      0.8
bare      0.1

PIPELINE_MEAN_AGB_2020: 53.4982697895214

GEDI_MEAN_AGB: 62.26774520873622

CALIBRATION_MULTIPLIER: 1.163920729655681

summary.csv:
                     metric         unit  ipcc_values  gedi_calibrated
                 valid area           ha      18771.1          18771.1
              CO2e stock t1       t CO2e    1733800.4        2018006.2
              CO2e stock t2       t CO2e    1745732.9        2031894.8
            net CO2e change       t CO2e      11932.5          13888.5
           annual CO2e rate    t CO2e/yr       2386.5           2777.7
         change per hectare    t CO2e/ha          0.6              0.7
change per hectare per year t CO2e/ha/yr          0.1              0.1

sensitivity_multiplier.csv:
       scenario  mult

In [17]:
import pandas as pd

print(pd.read_csv('summary.csv').to_string(index=False))
print()
print(pd.read_csv('sensitivity_multiplier.csv').to_string(index=False))
print()
print(pd.read_csv('sensitivity_ndvi_modifier.csv').to_string(index=False))

                     metric         unit  ipcc_values  gedi_calibrated
                 valid area           ha      18771.1          18771.1
              CO2e stock t1       t CO2e    1733800.4        2018006.2
              CO2e stock t2       t CO2e    1745732.9        2031894.8
            net CO2e change       t CO2e      11932.5          13888.5
           annual CO2e rate    t CO2e/yr       2386.5           2777.7
         change per hectare    t CO2e/ha          0.6              0.7
change per hectare per year t CO2e/ha/yr          0.1              0.1

       scenario  multiplier  stock_t1_tCO2e  stock_t2_tCO2e  net_change_tCO2e
    ipcc_values       0.800         1387040         1396586              9546
    ipcc_values       1.000         1733800         1745733             11933
    ipcc_values       1.200         2080560         2094880             14319
gedi_calibrated       0.931         1614405         1625516             11111
gedi_calibrated       1.164         20180

In [18]:
import os

os.chdir(os.path.join(os.path.expanduser('~'), 'co2_project'))
print(os.getcwd())

for f in sorted(os.listdir('.')):
    print(f, round(os.path.getsize(f) / 1024, 1), 'KB')

os.startfile('.')

C:\Users\mayar\co2_project
README.md 2.7 KB
report_numbers.txt 2.1 KB
sensitivity_forest_zone.csv 0.2 KB
sensitivity_multiplier.csv 0.3 KB
sensitivity_ndvi_modifier.csv 0.1 KB
summary.csv 0.3 KB
